In [2]:
import lightkurve as lk
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from astropy.timeseries import LombScargle
import astropy.units as u
import gyrointerp
from gyrointerp import gyro_age_posterior
from gyrointerp import get_summary_statistics

targets = pd.read_csv("C:\\Users\\smithlt\\Documents\\ASTR502\\ASTR502_Mega_Target_List.csv")

In [3]:
#convert txt file to csv
kepler_targets = pd.read_csv("C:\\Users\\smithlt\\Documents\\ASTR502\\Kepler_data.csv")

print(kepler_targets.head())

      TIC ID          2MASS ID  Kepler ID  Kep RA (J2000)  Kep Dec (J2000)  \
0  137220387  19261900+3802089    2853093      291.579170         38.03583   
1  271352188  19395364+4512492    8962094      294.973520         45.21369   
2  158935283  19161861+4600187    9458613      289.077540         46.00522   
3  184009076  19415549+4043551    5546761      295.481229         40.73198   
4  239225129  19472307+4008190    5031857      296.846160         40.13861   

                Condition flag     Period  Period Power  
0                    Exoplanet  26.611830      0.000033  
1                    Exoplanet  41.561185      0.000112  
2                    Exoplanet   2.721708      0.000129  
3  Eclipsing_binary; Exoplanet  11.019765      0.000146  
4          Planetary_candidate   3.542097      0.000147  


In [4]:
#create an array of the Kepler target TIC IDs
kepler_target_tic_ids = kepler_targets['TIC ID'].values
kepler_target_periods = kepler_targets['Period'].values

print(len(kepler_target_tic_ids))

1962


In [5]:
if 'mission_source' in targets.columns:
    print(f"\nMission sources in dataset:")
    print(targets['mission_source'].value_counts())

    # Filter for only Kepler targets
    kepler_targets_mega = targets[targets['mission_source'] == 'Kepler'].copy()
    print(f"\nFound {len(kepler_targets_mega)} Kepler targets!")

    # Clean tic_id: remove leading "TIC " if present and convert to numeric (coerce failures to NA)
    kepler_targets_mega['tic_id'] = kepler_targets_mega['tic_id'].astype(str).str.replace('TIC ', '', regex=False)
    kepler_targets_mega['tic_id'] = pd.to_numeric(kepler_targets_mega['tic_id'], errors='coerce').astype('Int64')
    print(kepler_targets_mega['tic_id'].head())

    # If you have a list/array of TIC IDs to match, filter to those; otherwise keep all Kepler targets
    if 'kepler_target_tic_ids' in globals():
        # ensure kepler_target_tic_ids are integers
        try:
            tic_list = [int(x) for x in kepler_target_tic_ids]
            print(len(tic_list))
        except Exception:
            tic_list = list(kepler_target_tic_ids)
        #should shorten to 1962 targets from the kepler targets csv
        matched = kepler_targets[kepler_targets['TIC ID'].isin(tic_list)].copy()
        matched_teff = kepler_targets_mega[kepler_targets_mega['tic_id'].isin(tic_list)].copy()
        print(len(matched))
        print(f"\nMatched {len(matched)} Kepler targets from provided TIC ID list.")

    # Build kepler_star_df with columns required downstream: 'target_name', 'tic_id', 'Teff', 'period'
    target_results = []
    for idx, r in matched.iterrows():
        tic = int(r['TIC ID']) if pd.notnull(r['TIC ID']) else None
        target_name = r.get('pl_name') or r.get('hostname') or f"TIC{tic}"
        #have to get teff from the mega target list
        teff = matched_teff[matched_teff['tic_id'] == tic]['st_teff'].values
        lit_age = matched_teff[matched_teff['tic_id'] == tic]['st_age'].values

        #want to find the single period value for this tic id
        period = kepler_targets['Period'][kepler_targets['TIC ID'] == tic].values

        target_results.append({
            'tic_ids': target_name,
            'Teff': teff,
            'period': period,
            'st_age': lit_age
        })

    # create DataFrame even if empty so later cells won't raise NameError
    kepler_star_df = pd.DataFrame(target_results, columns=['tic_ids', 'Teff', 'period', 'st_age'])
    print(f"\nFinal kepler_star_df has {len(kepler_star_df)} rows.")

else:
    # If no 'mission_source' column, create empty kepler_star_df to avoid NameError later
    print("No 'mission_source' in targets DataFrame; creating empty kepler_star_df.")
    kepler_star_df = pd.DataFrame(columns=['target_name', 'tic_ids', 'Teff', 'period'])
    print(kepler_star_df.head())


Mission sources in dataset:
mission_source
Kepler    2762
TESS       717
K2         548
WASP       168
HAT        139
Other      105
CoRoT       34
NGTS        22
KELT        21
Name: count, dtype: int64

Found 2762 Kepler targets!
1363    351766445
1364    351766604
1365    351766517
1366    351799800
1369    123126460
Name: tic_id, dtype: Int64
1962
1962

Matched 1962 Kepler targets from provided TIC ID list.

Final kepler_star_df has 1962 rows.


In [6]:
Teff = kepler_star_df['Teff']
print(Teff.head())
Prot = kepler_star_df['period']
print(Prot.head())
lit_age = kepler_star_df['st_age']
print(lit_age.head())

0                                    [5694.0]
1            [5739.0, 5739.0, 5739.0, 5739.0]
2    [5904.0, 5904.0, 5904.0, 5904.0, 5904.0]
3                                    [5683.0]
4                                    [6086.0]
Name: Teff, dtype: object
0    [26.61183003]
1    [41.56118472]
2    [2.721708315]
3    [11.01976462]
4    [3.542096695]
Name: period, dtype: object
0                            [4.07]
1          [1.62, 1.62, 1.62, 1.62]
2    [4.27, 4.27, 4.27, 4.27, 4.27]
3                            [4.17]
4                             [0.3]
Name: st_age, dtype: object


In [7]:
print(kepler_star_df.head())

        tic_ids                                      Teff         period  \
0  TIC137220387                                  [5694.0]  [26.61183003]   
1  TIC271352188          [5739.0, 5739.0, 5739.0, 5739.0]  [41.56118472]   
2  TIC158935283  [5904.0, 5904.0, 5904.0, 5904.0, 5904.0]  [2.721708315]   
3  TIC184009076                                  [5683.0]  [11.01976462]   
4  TIC239225129                                  [6086.0]  [3.542096695]   

                           st_age  
0                          [4.07]  
1        [1.62, 1.62, 1.62, 1.62]  
2  [4.27, 4.27, 4.27, 4.27, 4.27]  
3                          [4.17]  
4                           [0.3]  


In [ ]:
# calculate dictionary of summary statistics for each target and store results in results_df
# Note: gyro_age_posterior and get_summary_statistics were imported in earlier cells,
# so we don't re-import them here.

# ensure columns exist (store arrays as objects)
for col in ['age_grid', 'age_posterior', 'median', '+1sigma', '-1sigma', 'mean', 'mode']:
    if col not in kepler_star_df.columns:
        kepler_star_df[col] = [None] * len(kepler_star_df)

for i in range(kepler_star_df.shape[0]):
    Prot = kepler_star_df['period'].iloc[i]
    Prot_err = 0.2

    Teff = np.mean(kepler_star_df['Teff'].iloc[i])
    Teff_err = 100

    # uniformly spaced grid between 0 and 4000 megayears
    age_grid = np.linspace(0, 4000, 500)

    # calculate the age posterior at each age in `age_grid`
    age_posterior = gyro_age_posterior(
        Prot, Teff,
        Prot_err=Prot_err, Teff_err=Teff_err,
        age_grid=age_grid
    )

    # compute summary statistics
    result = get_summary_statistics(age_grid, age_posterior)

    # store results in the dataframe
    kepler_star_df.at[i, 'age_grid'] = age_grid
    kepler_star_df.at[i, 'age_posterior'] = age_posterior
    kepler_star_df.at[i, 'median'] = result.get('median', np.nan)
    kepler_star_df.at[i, '+1sigma'] = result.get('+1sigma', np.nan)
    kepler_star_df.at[i, '-1sigma'] = result.get('-1sigma', np.nan)
    kepler_star_df.at[i, 'mean'] = result.get('mean', np.nan)
    kepler_star_df.at[i, 'mode'] = result.get('mode', np.nan)

    print(f"\nTarget: {kepler_star_df['tic_ids'].iloc[i]}")
    print(f"Age = {result['median']} +{result['+1sigma']} -{result['-1sigma']} Myr.")
    print(f"Age (most probable): {age_grid[np.argmax(age_posterior)]:.1f} Myr")



Target: TIC137220387
Age = 3755.08 +168.44 -245.14 Myr.
Age (most probable): 3911.8 Myr

Target: TIC271352188
Age = 3995.92 +2.78 -2.78 Myr.
Age (most probable): 4000.0 Myr

Target: TIC158935283
Age = 83.9 +85.42 -57.31 Myr.
Age (most probable): 0.0 Myr

Target: TIC184009076
Age = 1087.76 +203.58 -120.35 Myr.
Age (most probable): 1042.1 Myr

Target: TIC239225129
Age = 295.59 +157.27 -114.76 Myr.
Age (most probable): 272.5 Myr

Target: TIC159521326
Age = 3261.58 +324.59 -324.81 Myr.
Age (most probable): 3262.5 Myr

Target: TIC27082352
Age = 3179.38 +302.79 -478.97 Myr.
Age (most probable): 3318.6 Myr

Target: TIC416283848
Age = 394.03 +266.95 -129.25 Myr.
Age (most probable): 336.7 Myr

Target: TIC164779321
Age = 75.6 +135.4 -51.62 Myr.
Age (most probable): 0.0 Myr

Target: TIC275486985
Age = 2267.52 +334.21 -254.58 Myr.
Age (most probable): 2172.3 Myr

Target: TIC270858032
Age = 61.93 +69.13 -42.28 Myr.
Age (most probable): 0.0 Myr

Target: TIC137342400
Age = nan +nan -nan Myr.
Age (m

c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\gyrointerp\gyro_posterior.py:479: RuntimeWarning: invalid value encountered in divide
  p_ages /= nptrapz(p_ages, age_grid)



Target: TIC138213510
Age = nan +nan -nan Myr.
Age (most probable): 0.0 Myr

Target: TIC158555483
Age = 77.43 +146.17 -52.87 Myr.
Age (most probable): 0.0 Myr

Target: TIC63065911
Age = 2238.97 +200.42 -156.76 Myr.
Age (most probable): 2196.4 Myr

Target: TIC48505882
Age = 2352.17 +195.44 -154.63 Myr.
Age (most probable): 2316.6 Myr

Target: TIC63006978
Age = 74.51 +129.63 -50.87 Myr.
Age (most probable): 0.0 Myr

Target: TIC279915745
Age = 3227.78 +279.65 -436.95 Myr.
Age (most probable): 3342.7 Myr

Target: TIC123445166
Age = 224.58 +93.13 -91.21 Myr.
Age (most probable): 224.4 Myr

Target: TIC275573748
Age = 3139.45 +343.27 -517.66 Myr.
Age (most probable): 3310.6 Myr

Target: TIC351908051
Age = 74.0 +126.32 -50.52 Myr.
Age (most probable): 0.0 Myr

Target: TIC158934047
Age = 72.05 +109.83 -49.19 Myr.
Age (most probable): 0.0 Myr

Target: TIC26541093
Age = 1081.04 +414.7 -182.89 Myr.
Age (most probable): 961.9 Myr

Target: TIC159575346
Age = 81.67 +80.1 -55.78 Myr.
Age (most probabl

c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\gyrointerp\gyro_posterior.py:479: RuntimeWarning: invalid value encountered in divide
  p_ages /= nptrapz(p_ages, age_grid)



Target: TIC158393923
Age = nan +nan -nan Myr.
Age (most probable): 0.0 Myr

Target: TIC158559861
Age = 75.58 +131.06 -51.61 Myr.
Age (most probable): 0.0 Myr

Target: TIC122446315
Age = 3526.98 +283.8 -352.31 Myr.
Age (most probable): 3607.2 Myr

Target: TIC271346561
Age = 872.7 +186.54 -109.63 Myr.
Age (most probable): 833.7 Myr

Target: TIC123417850
Age = nan +nan -nan Myr.
Age (most probable): 0.0 Myr

Target: TIC272707494
Age = 3995.96 +2.76 -2.76 Myr.
Age (most probable): 4000.0 Myr

Target: TIC267672183
Age = nan +nan -nan Myr.
Age (most probable): 0.0 Myr

Target: TIC63204017
Age = 3353.3 +303.73 -301.74 Myr.
Age (most probable): 3350.7 Myr

Target: TIC138214374
Age = 1099.1 +615.63 -263.88 Myr.
Age (most probable): 881.8 Myr

Target: TIC350738167
Age = 2149.94 +430.69 -317.49 Myr.
Age (most probable): 2004.0 Myr

Target: TIC158215023
Age = 1920.04 +698.74 -648.01 Myr.
Age (most probable): 1731.5 Myr


...


Target: TIC270787590
Age = nan +nan -nan Myr.
Age (most probable): 0.0 Myr

Target: TIC239306681
Age = 80.37 +70.79 -54.87 Myr.
Age (most probable): 0.0 Myr


c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\gyrointerp\gyro_posterior.py:479: RuntimeWarning: invalid value encountered in divide
  p_ages /= nptrapz(p_ages, age_grid)



Target: TIC164727404
Age = nan +nan -nan Myr.
Age (most probable): 0.0 Myr

Target: TIC164527535
Age = 3020.57 +298.55 -265.19 Myr.
Age (most probable): 2990.0 Myr

Target: TIC158394235
Age = 3980.13 +14.68 -31.1 Myr.
Age (most probable): 4000.0 Myr


c:\Users\smithlt\miniconda3\envs\astr502\Lib\site-packages\gyrointerp\gyro_posterior.py:479: RuntimeWarning: invalid value encountered in divide
  p_ages /= nptrapz(p_ages, age_grid)



Target: TIC272184502
Age = nan +nan -nan Myr.
Age (most probable): 0.0 Myr

Target: TIC271538318
Age = 2635.02 +526.14 -587.15 Myr.
Age (most probable): 2781.6 Myr

Target: TIC184428068
Age = 3976.51 +17.41 -37.36 Myr.
Age (most probable): 4000.0 Myr

Target: TIC240178935
Age = nan +nan -nan Myr.
Age (most probable): 0.0 Myr

Target: TIC120765183
Age = 2597.68 +347.18 -271.89 Myr.
Age (most probable): 2501.0 Myr

Target: TIC237159318
Age = 76.04 +177.21 -51.92 Myr.
Age (most probable): 0.0 Myr

Target: TIC268608781
Age = 3831.86 +121.09 -208.18 Myr.
Age (most probable): 4000.0 Myr

Target: TIC26491504
Age = 3995.99 +2.74 -2.74 Myr.
Age (most probable): 4000.0 Myr

Target: TIC121660143
Age = 2889.8 +253.45 -222.63 Myr.
Age (most probable): 2861.7 Myr

Target: TIC164555061
Age = 3819.33 +128.92 -208.19 Myr.
Age (most probable): 4000.0 Myr

Target: TIC158172348
Age = 3943.02 +42.34 -85.67 Myr.
Age (most probable): 4000.0 Myr

Target: TIC158494327
Age = 1114.6 +91.59 -90.16 Myr.
Age (most

In [ ]:
#match these with st_age (ages from the literature) to see how well they compare 
#k2_star_df  = k2_star_df.rename(columns={'tic_id': 'tic_ids'})
#k2_star_df = k2_star_df.astype({'tic_id': np.int64})
#k2_targets_mega = k2_targets_mega.astype({'tic_id': np.int64})
#k2_star_df = k2_star_df.merge(k2_targets_mega[["tic_id", "st_age"]], on="tic_id", how="left")


for i in range(len(kepler_star_df)):
    lit_age = kepler_star_df['st_age'].iloc[i]

    # Determine whether lit_age contains a usable value.
    # lit_age may be a scalar (float / pd.NA) or an array-like (np.ndarray, list, Series).
    if hasattr(lit_age, '__len__') and not np.isscalar(lit_age):
        # array-like: check length and that not all entries are NaN
        has_value = len(lit_age) > 0 and not np.all(pd.isna(lit_age))
        lit_age_repr = np.array2string(lit_age)
    else:
        # scalar: check for NaN / missing
        has_value = not pd.isna(lit_age)
        lit_age_repr = str(lit_age)

    if has_value:
        tic_col = 'tic_ids' if 'tic_ids' in kepler_star_df.columns else 'tic_id'
        derived_age = kepler_star_df['median'].iloc[i]
        print(f"Target: {kepler_star_df[tic_col].iloc[i]}, Literature Age: {lit_age_repr} Myr, Derived Age: {derived_age} Myr")

In [ ]:
import matplotlib.pyplot as plt

for i in range(len(kepler_star_df)):
    age_grid = kepler_star_df['age_grid'][i]
    age_posterior = kepler_star_df['age_posterior'][i]
    fig, ax = plt.subplots()
    ax.plot(age_grid, 1e3*age_posterior, c='k', lw=1)
    ax.update({
        'xlabel': 'Age [Myr]',
        'ylabel': 'Probability ($10^{-3}\,$Myr$^{-1}$)',
        'title': f'Prot = {Prot}d, Teff = {Teff}K',
        'xlim': [0,4000]
    })
    plt.show()